<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0;'>
  <h1 style='color: #fff; margin: 0 0 8px 0; font-size: 2.2em;'>🖼️ HYPIR Image Upscaler</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Google Colab Edition</h3>
  <p style='color: #aaa; margin: 0;'>Single-pass 4K restoration & upscaling | Diffusion-yielded score priors, SIGGRAPH 2025</p>
</div>

<p align="center">
  <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white"> /img</a>
  <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white"> /img</a>
  <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white"> /img</a>
  <a href="https://github.com/cafermutluozkan"><img src="https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white"> /img</a>
</p>

---

### 🚀 What is HYPIR?

**HYPIR** restores and upscales low-quality images in a **single forward pass** (no iterative diffusion sampling) by fine-tuning a Stable Diffusion 2.1 prior with adversarial training. Supports optional **text prompts** to guide fine detail generation, with an interactive before/after comparison slider.

| Feature | Detail |
|---|---|
| **Model** | HYPIR (SD 2.1 + LoRA, ~5 GB) |
| **Output** | Up to 4K (4096×4096), JPG |
| **License** | Non-commercial only (code + weights) — [commercial license available on request](https://github.com/XPixelGroup/HYPIR/blob/main/LICENSE) |
| **Paper** | [arXiv:2507.20590](https://arxiv.org/abs/2507.20590) (SIGGRAPH Asia 2025) |

### ⚡ Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Cell 1** = setup + model download (~5 min)
3. Run **Cell 2** = launch Gradio demo (public link)


In [ ]:
# @title ⚙️ Setup & Download Model (~5 GB)
import os, subprocess, sys
import torch

print('=== Colab Environment Check ===')
try:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
except Exception:
    print('WARNING: No GPU detected. Runtime → Change runtime type → T4 GPU')

print('[1/3] Installing dependencies...')
!pip install -q omegaconf python-dotenv accelerate diffusers lpips open_clip_torch peft einops pydantic opencv-python-headless timm vision-aided-loss polars tenacity huggingface_hub gradio nest_asyncio

# Old preinstalled Gradio + newer huggingface_hub can raise an "HfFolder" ImportError; force a modern Gradio.
try:
    import gradio as gr
    major = int(gr.__version__.split('.')[0])
except Exception:
    major = 0
if major < 5:
    print('Upgrading Gradio to a modern release...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'gradio'], check=True)

# Colab's preinstalled torchao is often too old for peft's version check, but
# upgrading it breaks the preinstalled transformers/diffusers (torchao's internal
# API changes between versions). HYPIR only uses peft's basic LoraConfig, which
# does not need torchao at all, so just remove it instead of upgrading it.
print('Removing torchao (not required by HYPIR, avoids transformers/diffusers version clashes)...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

print('[2/3] Cloning HYPIR...')
if not os.path.exists('HYPIR'):
    !git clone -q https://github.com/XPixelGroup/HYPIR.git
%cd HYPIR

print('[3/3] Downloading HYPIR_sd2.pth weights...')
if not os.path.exists('HYPIR_sd2.pth'):
    !wget -q https://huggingface.co/lxq007/HYPIR/resolve/main/HYPIR_sd2.pth

print('✓ Ready! Run the next cell to launch the Gradio demo.')

In [ ]:
# @title 🚀 Launch Gradio Demo
%cd /content/HYPIR

import random, asyncio, threading, asyncio.runners
import nest_asyncio
nest_asyncio.apply()

# Python 3.12+ changed asyncio.run()'s signature (loop_factory); keep nest_asyncio compatible.
_true_run = asyncio.runners.run
_patched_run = asyncio.run
def _compat_run(main, *, debug=False, **kwargs):
    if threading.current_thread() is not threading.main_thread():
        return _true_run(main, debug=debug, **kwargs)
    return _patched_run(main, debug=debug)
asyncio.run = _compat_run

# Some older Gradio builds import a removed huggingface_hub.utils.HfFolder; shim it defensively.
try:
    import huggingface_hub.utils as _hfu
    if not hasattr(_hfu, 'HfFolder'):
        class _DummyHfFolder:
            @staticmethod
            def get_token():
                return None
            @staticmethod
            def save_token(token):
                pass
            @staticmethod
            def delete_token():
                pass
        _hfu.HfFolder = _DummyHfFolder
except ImportError:
    pass

import os, sys, gc, tempfile, urllib.parse, uuid
import torch
import gradio as gr
import torchvision.transforms as transforms
from accelerate.utils import set_seed
from PIL import Image

sys.path.append(os.getcwd())
try:
    from HYPIR.enhancer.sd2 import SD2Enhancer
except ModuleNotFoundError:
    from enhancer.sd2 import SD2Enhancer

try:
    gr.close_all()
except Exception:
    pass

# Fallback error image if the repo's own placeholder is missing.
error_image_path = os.path.join('assets', 'gradio_error_img.png')
if os.path.exists(error_image_path):
    error_image = Image.open(error_image_path)
else:
    error_image = Image.new('RGB', (512, 512), (180, 50, 50))

MAX_SIZE = (4096, 4096)  # hard cap so a single T4/L4 never OOMs chasing an 8K output
to_tensor = transforms.ToTensor()

print('Loading HYPIR (Stable Diffusion 2.1 restoration LoRA)...')
model = SD2Enhancer(
    base_model_path='sd2-community/stable-diffusion-2-1-base',
    weight_path='HYPIR_sd2.pth',
    lora_modules=[
        'to_k', 'to_q', 'to_v', 'to_out.0',
        'conv', 'conv1', 'conv2', 'conv_shortcut', 'conv_out',
        'proj_in', 'proj_out', 'ff.net.2', 'ff.net.0.proj',
    ],
    lora_rank=256,
    model_t=200,
    coeff_t=200,
    device='cuda',
)
model.init_models()
print('✓ Model loaded on GPU.')


def create_comparison_slider(before_path, after_path):
    if not before_path or not after_path:
        return (
            "<div style='text-align:center;padding:40px;border:2px dashed #444;"
            "border-radius:12px;color:#888;font-family:sans-serif;'>"
            "🔍 Run restoration to see the before / after comparison slider."
            "</div>"
        )
    slider_id = f"slider-{uuid.uuid4().hex[:8]}"
    prefix = "/file=" if gr.__version__.startswith('4.') else "/gradio_api/file="
    before_url = f"{prefix}{urllib.parse.quote(before_path)}"
    after_url = f"{prefix}{urllib.parse.quote(after_path)}"
    return f"""
    <div id="{slider_id}" class="comparison-container" style="position:relative;width:100%;max-width:800px;height:500px;margin:0 auto;overflow:hidden;border-radius:12px;box-shadow:0 8px 30px rgba(0,0,0,0.4);background:#111;user-select:none;touch-action:none;">
      <img src="{after_url}" style="position:absolute;top:0;left:0;width:100%;height:100%;object-fit:contain;pointer-events:none;">
      <span style="position:absolute;bottom:16px;right:16px;background:rgba(0,0,0,0.8);color:#fff;padding:6px 14px;border-radius:6px;font-size:13px;font-weight:600;font-family:sans-serif;">Restored</span>
      <img class="clip-foreground" src="{before_url}" style="position:absolute;top:0;left:0;width:100%;height:100%;object-fit:contain;pointer-events:none;clip-path:inset(0 50% 0 0);z-index:2;">
      <span style="position:absolute;bottom:16px;left:16px;background:rgba(0,0,0,0.8);color:#fff;padding:6px 14px;border-radius:6px;font-size:13px;font-weight:600;font-family:sans-serif;">Original</span>
      <div class="clip-slider-handle" style="position:absolute;top:0;bottom:0;left:50%;width:4px;background:#7c5cff;cursor:ew-resize;z-index:5;transform:translateX(-50%);display:flex;align-items:center;justify-content:center;touch-action:none;">
        <div style="width:38px;height:38px;background:#fff;border:3px solid #7c5cff;border-radius:50%;display:flex;align-items:center;justify-content:center;box-shadow:0 4px 10px rgba(0,0,0,0.3);color:#7c5cff;font-weight:bold;font-size:18px;touch-action:none;">↔</div>
      </div>
    </div>
    """


def process(image, prompt, upscale, seed, progress=gr.Progress(track_tqdm=True)):
    if image is None:
        return None, None, "⚠️ Please upload an image."
    if seed == -1:
        seed = random.randint(0, 2**32 - 1)
    set_seed(int(seed))
    try:
        progress(0.1, desc='Preparing image...')
        img_rgb = image.convert('RGB')
        orig_w, orig_h = img_rgb.size
        out_w, out_h = int(orig_w * upscale), int(orig_h * upscale)
        max_w, max_h = MAX_SIZE
        if out_w > max_w or out_h > max_h:
            max_input_side = (int(max_w // upscale) // 16) * 16
            return error_image, create_comparison_slider(None, None), (
                f"❌ {out_w}x{out_h} exceeds the 4K safety limit ({max_w}x{max_h}). "
                f"Resize your input to at most {max_input_side}px on its longest side "
                f"for a {upscale}x upscale, then try again."
            )
        progress(0.3, desc='Restoring / upscaling...')
        image_tensor = to_tensor(img_rgb).unsqueeze(0)
        with torch.no_grad():
            enhanced = model.enhance(lq=image_tensor, prompt=prompt, upscale=upscale, return_type='pil')[0]
        progress(0.9, desc='Finalizing...')
        gc.collect()
        torch.cuda.empty_cache()
        out_dir = tempfile.mkdtemp()
        output_path = os.path.join(out_dir, 'restored_output.jpg')
        enhanced.save(output_path, format='JPEG', quality=100, subsampling=0)
        input_path = os.path.join(out_dir, 'original_input.jpg')
        img_rgb.save(input_path, format='JPEG', quality=100, subsampling=0)
        progress(1.0, desc='Done!')
        return output_path, create_comparison_slider(input_path, output_path), (
            f"✅ Done! Prompt: {prompt or '(none)'} · Seed: {seed}"
        )
    except Exception as exc:
        gc.collect()
        torch.cuda.empty_cache()
        return error_image, create_comparison_slider(None, None), f"❌ Failed: {exc}"


CSS = """
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align:center; background: linear-gradient(135deg,#0f0c29 0%,#302b63 55%,#24243e 100%); padding:28px; border-radius:15px; margin-bottom:20px; box-shadow:0 10px 25px rgba(0,0,0,0.4); }
.brand-title { color:#fff; font-size:2em; font-weight:700; margin:0 0 6px 0; }
.brand-subtitle { color:#c0c0ff; font-size:1em; margin:0; font-weight:400; }
button.primary { background: linear-gradient(135deg,#7c5cff 0%,#5b3fd9 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
"""

SLIDER_JS = """
() => {
    setTimeout(() => {
        const container = document.querySelector('.comparison-container');
        if (!container || container.dataset.initialized) return;
        container.dataset.initialized = "true";
        const foreground = container.querySelector('.clip-foreground');
        const handle = container.querySelector('.clip-slider-handle');
        const handleIcon = handle.querySelector('div');
        let isDragging = false;

        function setSliderPosition(clientX) {
            const rect = container.getBoundingClientRect();
            let pos = Math.min(1, Math.max(0, (clientX - rect.left) / rect.width));
            const pct = pos * 100;
            foreground.style.clipPath = `inset(0 ${100 - pct}% 0 0)`;
            handle.style.left = pct + "%";
        }

        function onDown(e) { isDragging = true; handleIcon.style.transform = 'scale(1.15)'; setSliderPosition(e.clientX); e.preventDefault(); }
        function onMove(e) { if (isDragging) setSliderPosition(e.clientX); }
        function onUp() { isDragging = false; handleIcon.style.transform = 'scale(1)'; }

        handle.addEventListener('pointerdown', onDown);
        window.addEventListener('pointermove', onMove);
        window.addEventListener('pointerup', onUp);
        container.addEventListener('pointerdown', (e) => {
            if (e.target !== handle && !handle.contains(e.target)) { setSliderPosition(e.clientX); onDown(e); }
        });
    }, 150);
}
"""

with gr.Blocks(css=CSS, title='HYPIR Image Upscaler — TheBuildAI') as demo:
    gr.HTML(
        "<div class='brand-header'>"
        "<div class='brand-title'>🖼️ HYPIR Image Upscaler</div>"
        "<div class='brand-subtitle'>Single-pass 4K restoration & upscaling — SIGGRAPH 2025</div>"
        "</div>"
    )
    gr.Markdown(
        "Upload an image, optionally describe details to guide the restoration "
        "(e.g. *sharp facial features, fine skin texture*), pick an upscale factor, "
        "and click **Restore & Upscale**."
    )
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label='Input image', type='pil')
            prompt_input = gr.Textbox(label='Prompt (optional, guides detail generation)', placeholder='e.g. sharp portrait photo, fine detail')
            upscale_slider = gr.Slider(1, 8, value=2, step=1, label='Upscale factor')
            seed_input = gr.Number(label='Seed (-1 for random)', value=-1)
            with gr.Row():
                gen_btn = gr.Button('✨ Restore & Upscale', variant='primary', size='lg')
                stop_btn = gr.Button('🛑 Stop', size='lg')
                clear_btn = gr.Button('🗑️ Clear', size='lg')
        with gr.Column():
            output_image = gr.Image(label='Restored output', type='filepath')
            status_output = gr.Textbox(label='Status', interactive=False)
    comparison_slider = gr.HTML(value=create_comparison_slider(None, None))

    gen_event = gen_btn.click(
        fn=process,
        inputs=[input_image, prompt_input, upscale_slider, seed_input],
        outputs=[output_image, comparison_slider, status_output],
    )
    gen_event.then(fn=None, js=SLIDER_JS)
    stop_btn.click(fn=None, cancels=[gen_event])
    clear_btn.click(
        fn=lambda: (None, '', 2, -1, None, create_comparison_slider(None, None), ''),
        outputs=[input_image, prompt_input, upscale_slider, seed_input, output_image, comparison_slider, status_output],
    )

demo.queue()
demo.launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,
    allowed_paths=[tempfile.gettempdir(), os.getcwd()],
)

---

<div align='center'>

### 🎉 Enjoyed this notebook?

If this was helpful, please **upvote** and subscribe for more free AI tools!

  <a href='https://www.youtube.com/@thebuildai?sub_confirmation=1'>
    <img src='https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white' />
  </a>
  <a href='https://www.instagram.com/thebuildai/'>
    <img src='https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white' />
  </a>
  <a href='https://www.tiktok.com/@the.build.ai'>
    <img src='https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white' />
  </a>
  <a href='https://github.com/cafermutluozkan'>
    <img src='https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white' />
  </a>

  <p style='color: #888; margin-top: 15px;'>Built by <strong><a href='https://www.thebuildai.tech/'>TheBuildAI</a></strong> 🌍</p>

</div>